##DATASET


In [ ]:
import os
import pandas as pd
import kagglehub

# Download the complete dataset
dataset_path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Dataset path:", dataset_path)
print("Files:", os.listdir(dataset_path))

Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
Dataset path: /kaggle/input/network-intrusion-dataset
Files: ['Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']


Load into csv files

In [ ]:
import os
import pandas as pd

datasets = {}

for file in os.listdir(dataset_path):
    if file.endswith(".csv"):
        file_path = os.path.join(dataset_path, file)
        print(f"Loading {file}...")
        datasets[file] = pd.read_csv(file_path)

print("Loaded", len(datasets), "CSV files.")

Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
Loading Tuesday-WorkingHours.pcap_ISCX.csv...
Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
Loading Monday-WorkingHours.pcap_ISCX.csv...
Loading Friday-WorkingHours-Morning.pcap_ISCX.csv...
Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
Loading Wednesday-workingHours.pcap_ISCX.csv...
Loaded 8 CSV files.


Combined data into one

In [ ]:
import os
import pandas as pd

datasets = {}

for file in os.listdir(dataset_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(dataset_path, file))
        df["source_file"] = file      # Add source filename
        datasets[file] = df

combined_df = pd.concat(
    datasets.values(),
    ignore_index=True
)

print(combined_df.shape)
print(combined_df.head())

# Check records from each file
print(combined_df["source_file"].value_counts())

(2830743, 80)
    Destination Port   Flow Duration   Total Fwd Packets  \
0                 22         1266342                  41   
1                 22         1319353                  41   
2                 22             160                   1   
3                 22         1303488                  41   
4              35396              77                   1   

    Total Backward Packets  Total Length of Fwd Packets  \
0                       44                         2664   
1                       44                         2664   
2                        1                            0   
3                       42                         2728   
4                        2                            0   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                          6954                     456   
1                          6954                     456   
2                             0                       0   
3                          6634   

##Clean + Preprocess (Scaling, Encoding, SMOTE)


In [ ]:
# ── STEP 1: Strip column names & drop source_file ─────────────────────────────
combined_df.columns = combined_df.columns.str.strip()
combined_df.drop(columns=["source_file"], inplace=True)

# ── STEP 2: Replace Inf/-Inf with NaN, then drop ──────────────────────────────
import numpy as np
combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)
combined_df.dropna(inplace=True)

print("After cleaning:", combined_df.shape)
print("\nLabel distribution before sampling:")
print(combined_df["Label"].value_counts())

After cleaning: (2827876, 79)

Label distribution before sampling:
Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import LabelEncoder

# ── STEP 3: Cap BENIGN + keep all attacks ─────────────────────────────────────
BENIGN_CAP = 200_000  # roughly 2x the next biggest class, still representative

df_benign = combined_df[combined_df["Label"] == "BENIGN"].sample(
    n=BENIGN_CAP, random_state=42
)
df_attacks = combined_df[combined_df["Label"] != "BENIGN"]

df = pd.concat([df_benign, df_attacks], ignore_index=True).sample(
    frac=1, random_state=42  # shuffle
).reset_index(drop=True)

print("Final shape:", df.shape)
print("\nLabel distribution after sampling:")
print(df["Label"].value_counts())

# ── STEP 4: Encode Label ───────────────────────────────────────────────────────
le = LabelEncoder()
df["Label_enc"] = le.fit_transform(df["Label"])

print("\nLabel encoding map:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")

Final shape: (756556, 79)

Label distribution after sampling:
Label
DoS Hulk                      230124
BENIGN                        200000
PortScan                      158804
DDoS                          128025
DoS GoldenEye                  10293
FTP-Patator                     7935
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1956
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Infiltration                      36
Web Attack � Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64

Label encoding map:
  0 → BENIGN
  1 → Bot
  2 → DDoS
  3 → DoS GoldenEye
  4 → DoS Hulk
  5 → DoS Slowhttptest
  6 → DoS slowloris
  7 → FTP-Patator
  8 → Heartbleed
  9 → Infiltration
  10 → PortScan
  11 → SSH-Patator
  12 → Web Attack � Brute Force
  13 → Web Attack � Sql Injection
  14 → Web Attack � XSS


#Feature Selection

In [ ]:
# ── STEP 5: Feature Selection — Correlation Matrix ────────────────────────────

X = df.drop(columns=["Label", "Label_enc"])
y = df["Label_enc"]

# 5a. Drop zero-variance columns (same value in every row — useless)
zero_var = [col for col in X.columns if X[col].nunique() <= 1]
X.drop(columns=zero_var, inplace=True)
print(f"Dropped {len(zero_var)} zero-variance columns: {zero_var}")

# 5b. Drop highly correlated features (>0.95)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
X.drop(columns=high_corr, inplace=True)
print(f"Dropped {len(high_corr)} highly correlated columns: {high_corr}")

print(f"\nFinal feature set: {X.shape[1]} features")
print(X.columns.tolist())

Dropped 8 zero-variance columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Dropped 30 highly correlated columns: ['Total Backward Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Std', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Std', 'Fwd IAT Max', 'Bwd IAT Max', 'Fwd Packets/s', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'SYN Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Idle Mean', 'Idle Max', 'Idle Min']

Final feature set: 40 features
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fw

##TRAIN MODEL(R.F. + XGBOOST+ I.F.)

In [ ]:
from sklearn.model_selection import train_test_split

# ── Train/Test Split ───────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # preserves class proportions even for rare classes
)

print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"\nClass distribution in train:")
print(y_train.value_counts().sort_index().rename(index=dict(enumerate(le.classes_))))

Train: (605244, 40)
Test:  (151312, 40)

Class distribution in train:
Label_enc
BENIGN                        160000
Bot                             1565
DDoS                          102420
DoS GoldenEye                   8234
DoS Hulk                      184099
DoS Slowhttptest                4399
DoS slowloris                   4637
FTP-Patator                     6348
Heartbleed                         9
Infiltration                      29
PortScan                      127043
SSH-Patator                     4717
Web Attack � Brute Force        1205
Web Attack � Sql Injection        17
Web Attack � XSS                 522
Name: count, dtype: int64


##RANDOM FOREST


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import time

rf = RandomForestClassifier(
    n_estimators=200,        # more trees = more stable, still fast with n_jobs=-1
    class_weight="balanced", # handles Heartbleed/Infiltration with 11/36 samples
    max_depth=None,          # let trees grow fully — RF handles overfitting via bagging
    min_samples_leaf=2,      # slight regularization, avoids single-sample leaves
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
rf.fit(X_train, y_train)
print(f"Training time: {time.time()-t0:.1f}s")

y_pred_rf = rf.predict(X_test)

print("\n── Random Forest Classification Report ──")
print(classification_report(
    y_test, y_pred_rf,
    target_names=le.classes_,
    digits=4               # 4 decimal places — you'll want precision for comparison
))

# FPR — critical for SOC
cm = confusion_matrix(y_test, y_pred_rf)
y_binary      = (y_test != 0).astype(int)
y_pred_binary = (y_pred_rf != 0).astype(int)
TN = ((y_binary == 0) & (y_pred_binary == 0)).sum()
FP = ((y_binary == 0) & (y_pred_binary == 1)).sum()
print(f"False Positive Rate: {FP/(FP+TN):.4f}")
print(f"(FP={FP}, TN={TN} — out of {TN+FP} BENIGN test samples)")

Training time: 412.2s

── Random Forest Classification Report ──
                            precision    recall  f1-score   support

                    BENIGN     0.9991    0.9985    0.9988     40000
                       Bot     0.9530    0.9847    0.9686       391
                      DDoS     1.0000    0.9999    0.9999     25605
             DoS GoldenEye     0.9971    0.9985    0.9978      2059
                  DoS Hulk     0.9994    0.9998    0.9996     46025
          DoS Slowhttptest     0.9954    0.9918    0.9936      1100
             DoS slowloris     0.9948    0.9957    0.9953      1159
               FTP-Patator     1.0000    1.0000    1.0000      1587
                Heartbleed     1.0000    1.0000    1.0000         2
              Infiltration     1.0000    0.1429    0.2500         7
                  PortScan     0.9995    0.9995    0.9995     31761
               SSH-Patator     1.0000    1.0000    1.0000      1180
  Web Attack � Brute Force     0.7468    0.7815   

#XGBOOST

In [ ]:
from xgboost import XGBClassifier
import time

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,      # slower learning = better generalization
    max_depth=8,             # deeper than default — network data has complex interactions
    subsample=0.8,           # row sampling per tree — reduces overfitting
    colsample_bytree=0.8,    # feature sampling per tree
    min_child_weight=3,      # helps with rare classes — needs min 3 samples per leaf
    scale_pos_weight=1,      # we'll handle imbalance via sample_weight instead
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=42,
    verbosity=1
)

# Compute sample weights — gives rare classes more influence than class_weight param
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

t0 = time.time()
xgb.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=50               # print loss every 50 rounds so you can watch progress
)
print(f"\nTraining time: {time.time()-t0:.1f}s")

y_pred_xgb = xgb.predict(X_test)

print("\n── XGBoost Classification Report ──")
print(classification_report(
    y_test, y_pred_xgb,
    target_names=le.classes_,
    digits=4
))

# FPR
y_binary      = (y_test != 0).astype(int)
y_pred_binary = (y_pred_xgb != 0).astype(int)
TN = ((y_binary == 0) & (y_pred_binary == 0)).sum()
FP = ((y_binary == 0) & (y_pred_binary == 1)).sum()
print(f"False Positive Rate: {FP/(FP+TN):.4f}")
print(f"(FP={FP}, TN={TN} — out of {TN+FP} BENIGN test samples)")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [04:13:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-mlogloss:2.35842
[50]	validation_0-mlogloss:0.15528
[100]	validation_0-mlogloss:0.02135
[150]	validation_0-mlogloss:0.00802
[200]	validation_0-mlogloss:0.00623
[250]	validation_0-mlogloss:0.00583
[299]	validation_0-mlogloss:0.00575

Training time: 469.4s

── XGBoost Classification Report ──
                            precision    recall  f1-score   support

                    BENIGN     0.9999    0.9984    0.9992     40000
                       Bot     0.9443    0.9974    0.9701       391
                      DDoS     0.9996    0.9999    0.9998     25605
             DoS GoldenEye     0.9866    0.9995    0.9930      2059
                  DoS Hulk     0.9997    0.9994    0.9995     46025
          DoS Slowhttptest     0.9928    0.9964    0.9946      1100
             DoS slowloris     0.9957    0.9965    0.9961      1159
               FTP-Patator     1.0000    1.0000    1.0000      1587
                Heartbleed     1.0000    1.0000    1.0000         2
          

#ISOLATION FOREST

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import numpy as np
import time

# ── FIX 1: Top 15 features using RF importances ───────────────────────────────
feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top15_features = feature_importances.nlargest(15).index.tolist()

print("Top 15 features selected:")
for f, v in feature_importances[top15_features].sort_values(ascending=False).items():
    print(f"  {f}: {v:.4f}")

X_train_top = X_train[top15_features]
X_test_top  = X_test[top15_features]

# ── FIX 2: Tightest BENIGN samples only ───────────────────────────────────────
X_train_benign_top = X_train_top[y_train == 0]
print(f"\nBENIGN training samples available: {X_train_benign_top.shape[0]}")

# Find 80K rows closest to centroid = most "typical" normal traffic
scaler   = RobustScaler()
X_scaled = scaler.fit_transform(X_train_benign_top)
centroid  = X_scaled.mean(axis=0)
distances = np.linalg.norm(X_scaled - centroid, axis=1)
tight_idx = np.argsort(distances)[:80_000]

X_train_tight = X_train_benign_top.iloc[tight_idx]
print(f"Training on {X_train_tight.shape[0]} tightest BENIGN samples")

# ── FIX 3: Contamination matches real attack ratio ────────────────────────────
y_binary = (y_test != 0).astype(int)
attack_ratio = y_binary.sum() / len(y_binary)
print(f"True attack ratio in test set: {attack_ratio:.3f}")

# ── Train ──────────────────────────────────────────────────────────────────────
# contamination max is 0.5 in sklearn — use 0.5 (most aggressive allowed)
iso = IsolationForest(
    n_estimators=300,
    contamination=0.5,   # sklearn hard cap is 0.5
    max_samples=1.0,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)

t0 = time.time()
iso.fit(X_train_tight)
print(f"Training time: {time.time()-t0:.1f}s")

iso_raw    = iso.predict(X_test_top)
iso_binary = np.where(iso_raw == -1, 1, 0)
y_binary   = (y_test != 0).astype(int)

print("\n── Isolation Forest Report (BENIGN vs ATTACK) ──")
print(classification_report(
    y_binary, iso_binary,
    target_names=["BENIGN", "ATTACK"],
    digits=4
))

TN = ((y_binary == 0) & (iso_binary == 0)).sum()
FP = ((y_binary == 0) & (iso_binary == 1)).sum()
FN = ((y_binary == 1) & (iso_binary == 0)).sum()
TP = ((y_binary == 1) & (iso_binary == 1)).sum()

print(f"False Positive Rate : {FP/(FP+TN):.4f}")
print(f"TP={TP}  FP={FP}  TN={TN}  FN={FN}")

iso_scores = iso.decision_function(X_test_top)
print(f"Anomaly score range : {iso_scores.min():.4f} to {iso_scores.max():.4f}")

Top 15 features selected:
  Destination Port: 0.0989
  Init_Win_bytes_backward: 0.0807
  Fwd Packet Length Max: 0.0487
  Bwd Packet Length Max: 0.0449
  Bwd Packets/s: 0.0449
  Flow Packets/s: 0.0409
  Init_Win_bytes_forward: 0.0399
  Total Length of Fwd Packets: 0.0397
  Packet Length Variance: 0.0387
  Flow IAT Mean: 0.0385
  Bwd Header Length: 0.0383
  Flow Duration: 0.0380
  Flow Bytes/s: 0.0357
  Fwd Packet Length Mean: 0.0354
  Flow IAT Std: 0.0337

BENIGN training samples available: 160000
Training on 80000 tightest BENIGN samples
True attack ratio in test set: 0.736
Training time: 7.2s

── Isolation Forest Report (BENIGN vs ATTACK) ──
              precision    recall  f1-score   support

      BENIGN     0.9215    0.3112    0.4653     40000
      ATTACK     0.8001    0.9905    0.8851    111312

    accuracy                         0.8109    151312
   macro avg     0.8608    0.6508    0.6752    151312
weighted avg     0.8322    0.8109    0.7741    151312

False Positive Rate : 

##SAVE INTO FOL

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies & create folder structure
# ══════════════════════════════════════════════════════════════════════════════
import os

ROOT = "/content/AI-Network-Intrusion-Detection-System"

DIRS = [
    f"{ROOT}/data/raw",
    f"{ROOT}/data/processed",
    f"{ROOT}/data/sample",
    f"{ROOT}/notebooks",
    f"{ROOT}/src/preprocessing",
    f"{ROOT}/src/models",
    f"{ROOT}/src/evaluation",
    f"{ROOT}/src/dashboard/pages",
    f"{ROOT}/src/dashboard/components",
    f"{ROOT}/src/dashboard/assets",
    f"{ROOT}/src/utils",
    f"{ROOT}/models",
    f"{ROOT}/reports/figures",
    f"{ROOT}/reports/metrics",
    f"{ROOT}/dashboard_screenshots",
    f"{ROOT}/tests",
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)

print("✅ Folder structure created at:", ROOT)

✅ Folder structure created at: /content/AI-Network-Intrusion-Detection-System


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Save Models
# ══════════════════════════════════════════════════════════════════════════════
import joblib

MODELS = f"{ROOT}/models"

# Random Forest
joblib.dump(rf, f"{MODELS}/random_forest.pkl")
print("✅ Saved: random_forest.pkl")

# XGBoost (native JSON format — smaller & faster than pickle)
xgb.save_model(f"{MODELS}/xgboost.json")
print("✅ Saved: xgboost.json")

# Isolation Forest
joblib.dump(iso, f"{MODELS}/isolation_forest.pkl")
print("✅ Saved: isolation_forest.pkl")

# RobustScaler (used inside Isolation Forest pipeline)
joblib.dump(scaler, f"{MODELS}/robust_scaler.pkl")
print("✅ Saved: robust_scaler.pkl")

# LabelEncoder — needed to decode predictions back to class names
joblib.dump(le, f"{MODELS}/label_encoder.pkl")
print("✅ Saved: label_encoder.pkl")

# Selected feature list — top-15 used by Isolation Forest, and full feature set
import json
joblib.dump(X.columns.tolist(), f"{MODELS}/selected_features.pkl")
joblib.dump(top15_features,     f"{MODELS}/top15_features.pkl")

# Also save as readable JSON for inspection without Python
with open(f"{MODELS}/selected_features.json", "w") as f_:
    json.dump({"all_features": X.columns.tolist(), "top15_iso": top15_features}, f_, indent=2)

print("✅ Saved: selected_features.pkl / top15_features.pkl / selected_features.json")

# File sizes
for fname in os.listdir(MODELS):
    size_mb = os.path.getsize(f"{MODELS}/{fname}") / 1e6
    print(f"   {fname:40s} {size_mb:.1f} MB")

✅ Saved: random_forest.pkl
✅ Saved: xgboost.json
✅ Saved: isolation_forest.pkl
✅ Saved: robust_scaler.pkl
✅ Saved: label_encoder.pkl
✅ Saved: selected_features.pkl / top15_features.pkl / selected_features.json
   selected_features.json                   0.0 MB
   isolation_forest.pkl                     94.8 MB
   selected_features.pkl                    0.0 MB
   random_forest.pkl                        114.3 MB
   xgboost.json                             15.6 MB
   robust_scaler.pkl                        0.0 MB
   top15_features.pkl                       0.0 MB
   label_encoder.pkl                        0.0 MB


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Save Processed Data
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd

DATA = f"{ROOT}/data"

# 3a. combined.csv — all raw files merged, before cleaning
print("Saving combined.csv ...", end=" ")
combined_df.to_csv(f"{DATA}/processed/combined.csv", index=False)
print(f"✅  shape={combined_df.shape}")

# 3b. cleaned.csv — after NaN/Inf removal, BENIGN cap, label encoding
print("Saving cleaned.csv ...", end=" ")
df.to_csv(f"{DATA}/processed/cleaned.csv", index=False)
print(f"✅  shape={df.shape}")

# 3c. train.csv & test.csv — the exact splits used for training
print("Saving train.csv ...", end=" ")
train_df = X_train.copy()
train_df["Label_enc"] = y_train.values
train_df["Label"]     = le.inverse_transform(y_train.values)
train_df.to_csv(f"{DATA}/processed/train.csv", index=False)
print(f"✅  shape={train_df.shape}")

print("Saving test.csv ...", end=" ")
test_df = X_test.copy()
test_df["Label_enc"] = y_test.values
test_df["Label"]     = le.inverse_transform(y_test.values)
test_df.to_csv(f"{DATA}/processed/test.csv", index=False)
print(f"✅  shape={test_df.shape}")

# 3d. sample_flows.csv — 200 rows for demo/dashboard
print("Saving sample_flows.csv ...", end=" ")
import numpy as np

# Run predictions on the sample so the CSV is dashboard-ready
sample_X   = X_test.sample(n=200, random_state=42)
sample_rf  = le.inverse_transform(rf.predict(sample_X))
sample_xgb = le.inverse_transform(xgb.predict(sample_X))
sample_iso = iso.decision_function(sample_X[top15_features])
sample_true= le.inverse_transform(y_test.loc[sample_X.index].values)

sample_df = sample_X.copy()
sample_df["True_Label"]    = sample_true
sample_df["RF_Pred"]       = sample_rf
sample_df["XGB_Pred"]      = sample_xgb
sample_df["ISO_Score"]     = sample_iso
sample_df["Zero_Day_Flag"] = (
    (sample_df["RF_Pred"] == "BENIGN") &
    (sample_df["XGB_Pred"] == "BENIGN") &
    (sample_df["ISO_Score"] < -0.5)
)
sample_df.to_csv(f"{DATA}/sample/sample_flows.csv", index=False)
print(f"✅  {len(sample_df)} rows, {sample_df['Zero_Day_Flag'].sum()} zero-day flags")

Saving combined.csv ... ✅  shape=(2827876, 79)
Saving cleaned.csv ... ✅  shape=(756556, 80)
Saving train.csv ... ✅  shape=(605244, 42)
Saving test.csv ... ✅  shape=(151312, 42)
Saving sample_flows.csv ... ✅  200 rows, 0 zero-day flags


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Save Metrics CSVs
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    precision_score, recall_score
)

METRICS = f"{ROOT}/reports/metrics"

# ── Random Forest metrics ──────────────────────────────────────────────────
y_pred_rf_enc  = rf.predict(X_test)
y_pred_rf_lbl  = le.inverse_transform(y_pred_rf_enc)
y_test_lbl     = le.inverse_transform(y_test.values)

y_bin_true = (y_test != 0).astype(int)
y_bin_rf   = (y_pred_rf_enc != 0).astype(int)
TN = ((y_bin_true == 0) & (y_bin_rf == 0)).sum()
FP = ((y_bin_true == 0) & (y_bin_rf == 1)).sum()

rf_metrics = {
    "accuracy":   accuracy_score(y_test, y_pred_rf_enc),
    "macro_f1":   f1_score(y_test, y_pred_rf_enc, average="macro"),
    "macro_prec": precision_score(y_test, y_pred_rf_enc, average="macro", zero_division=0),
    "macro_rec":  recall_score(y_test, y_pred_rf_enc, average="macro", zero_division=0),
    "FPR":        FP / (FP + TN) if (FP + TN) > 0 else 0,
}
pd.DataFrame([rf_metrics]).to_csv(f"{METRICS}/rf_metrics.csv", index=False)
print("✅ rf_metrics.csv")

# Per-class report
rf_report = pd.DataFrame(
    classification_report(y_test_lbl, y_pred_rf_lbl, output_dict=True)
).T.reset_index().rename(columns={"index": "class"})
rf_report.to_csv(f"{METRICS}/rf_class_report.csv", index=False)
print("✅ rf_class_report.csv")

# ── XGBoost metrics ───────────────────────────────────────────────────────
y_pred_xgb_enc = xgb.predict(X_test)
y_pred_xgb_lbl = le.inverse_transform(y_pred_xgb_enc)

y_bin_xgb = (y_pred_xgb_enc != 0).astype(int)
TN_x = ((y_bin_true == 0) & (y_bin_xgb == 0)).sum()
FP_x = ((y_bin_true == 0) & (y_bin_xgb == 1)).sum()

xgb_metrics = {
    "accuracy":   accuracy_score(y_test, y_pred_xgb_enc),
    "macro_f1":   f1_score(y_test, y_pred_xgb_enc, average="macro"),
    "macro_prec": precision_score(y_test, y_pred_xgb_enc, average="macro", zero_division=0),
    "macro_rec":  recall_score(y_test, y_pred_xgb_enc, average="macro", zero_division=0),
    "FPR":        FP_x / (FP_x + TN_x) if (FP_x + TN_x) > 0 else 0,
}
pd.DataFrame([xgb_metrics]).to_csv(f"{METRICS}/xgb_metrics.csv", index=False)
print("✅ xgb_metrics.csv")

xgb_report = pd.DataFrame(
    classification_report(y_test_lbl, y_pred_xgb_lbl, output_dict=True)
).T.reset_index().rename(columns={"index": "class"})
xgb_report.to_csv(f"{METRICS}/xgb_class_report.csv", index=False)
print("✅ xgb_class_report.csv")

# ── Isolation Forest metrics ──────────────────────────────────────────────
from sklearn.metrics import classification_report as cr

iso_raw    = iso.predict(X_test[top15_features])
iso_binary = np.where(iso_raw == -1, 1, 0)
iso_scores = iso.decision_function(X_test[top15_features])
y_bin_test = (y_test != 0).astype(int)

TN_i = ((y_bin_test == 0) & (iso_binary == 0)).sum()
FP_i = ((y_bin_test == 0) & (iso_binary == 1)).sum()
FN_i = ((y_bin_test == 1) & (iso_binary == 0)).sum()
TP_i = ((y_bin_test == 1) & (iso_binary == 1)).sum()

iso_metrics = {
    "TP": int(TP_i), "FP": int(FP_i), "TN": int(TN_i), "FN": int(FN_i),
    "recall_attack":    TP_i / (TP_i + FN_i) if (TP_i + FN_i) > 0 else 0,
    "precision_attack": TP_i / (TP_i + FP_i) if (TP_i + FP_i) > 0 else 0,
    "FPR":              FP_i / (FP_i + TN_i) if (FP_i + TN_i) > 0 else 0,
    "score_min":  float(iso_scores.min()),
    "score_max":  float(iso_scores.max()),
    "score_mean": float(iso_scores.mean()),
}
pd.DataFrame([iso_metrics]).to_csv(f"{METRICS}/iso_metrics.csv", index=False)
print("✅ iso_metrics.csv")

print("\nAll metrics saved.")

✅ rf_metrics.csv
✅ rf_class_report.csv
✅ xgb_metrics.csv
✅ xgb_class_report.csv
✅ iso_metrics.csv

All metrics saved.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Save Figures (PNG)
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — works in Colab headlessly
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sns

FIGS = f"{ROOT}/reports/figures"
DARK = "#0a0d14"
plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": "#111520",
    "axes.edgecolor": "#1e2740", "axes.labelcolor": "#e2e8f0",
    "xtick.color": "#64748b", "ytick.color": "#64748b",
    "text.color": "#e2e8f0", "grid.color": "#1e2740",
    "font.family": "monospace"
})

# ── 5a. Attack Distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
label_counts = df["Label"].value_counts()
colors = ["#22c55e" if l == "BENIGN" else "#ef4444" for l in label_counts.index]
bars = ax.barh(label_counts.index, label_counts.values, color=colors, height=0.6)
ax.set_xlabel("Sample Count")
ax.set_title("Attack Distribution (after BENIGN cap)", pad=12, fontsize=13)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, label_counts.values):
    ax.text(val + max(label_counts)*0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9, color="#94a3b8")
plt.tight_layout()
plt.savefig(f"{FIGS}/attack_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ attack_distribution.png")

# ── 5b. Confusion Matrix — Random Forest ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(
    y_test_lbl, y_pred_rf_lbl,
    display_labels=le.classes_,
    ax=ax, colorbar=False,
    cmap="Blues",
    xticks_rotation=45
)
ax.set_title("Random Forest — Confusion Matrix", pad=12, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGS}/confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ confusion_matrix_rf.png")

# ── 5c. Confusion Matrix — XGBoost ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(
    y_test_lbl, y_pred_xgb_lbl,
    display_labels=le.classes_,
    ax=ax, colorbar=False,
    cmap="Purples",
    xticks_rotation=45
)
ax.set_title("XGBoost — Confusion Matrix", pad=12, fontsize=13)
plt.tight_layout()
plt.savefig(f"{FIGS}/confusion_matrix_xgb.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ confusion_matrix_xgb.png")

# ── 5d. Feature Importance (RF) ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,6))

fi = pd.Series(
    rf.feature_importances_,
    index=X.columns
).nlargest(15).sort_values()

colors = [
    "#7c3aed" if i < len(fi)-3 else "#00d4ff"
    for i in range(len(fi))
]

ax.barh(
    fi.index,
    fi.values,
    color=colors,
    height=0.7
)

ax.set_xlabel("Importance")
ax.set_title("Random Forest — Top 15 Feature Importance")

plt.tight_layout()
plt.savefig(f"{FIGS}/feature_importance.png", dpi=150)
plt.close()

# ── 5e. Anomaly Score Histogram ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
y_bin_test = (y_test != 0).astype(int)
ax.hist(iso_scores[y_bin_test == 0], bins=80, alpha=0.7, color="#22c55e", label="BENIGN",  density=True)
ax.hist(iso_scores[y_bin_test == 1], bins=80, alpha=0.7, color="#ef4444", label="ATTACK",  density=True)
ax.axvline(x=0, color="#eab308", linestyle="--", linewidth=1.2, label="Decision boundary")
ax.set_xlabel("Isolation Forest Score")
ax.set_ylabel("Density")
ax.set_title("Anomaly Score Distribution — Isolation Forest", pad=12, fontsize=13)
ax.legend(framealpha=0.2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGS}/anomaly_score_histogram.png", dpi=150, bbox_inches="tight")
plt.close()
print("✅ anomaly_score_histogram.png")

print("\nAll figures saved.")

✅ attack_distribution.png
✅ confusion_matrix_rf.png
✅ confusion_matrix_xgb.png
✅ anomaly_score_histogram.png

All figures saved.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Copy raw CSVs into data/raw/
# ══════════════════════════════════════════════════════════════════════════════
import shutil

RAW_DST = f"{ROOT}/data/raw"

# dataset_path is the path returned by kagglehub.dataset_download(...)
# — it should already be defined from your training notebook
copied = 0
for fname in os.listdir(dataset_path):
    if fname.endswith(".csv"):
        src = os.path.join(dataset_path, fname)
        dst = os.path.join(RAW_DST, fname)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f"  ✅ {fname}  ({size_mb:.0f} MB)")
        copied += 1

print(f"\nCopied {copied} raw CSV files → {RAW_DST}")

  ✅ Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  (77 MB)
  ✅ Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  (52 MB)
  ✅ Tuesday-WorkingHours.pcap_ISCX.csv  (135 MB)
  ✅ Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  (77 MB)
  ✅ Monday-WorkingHours.pcap_ISCX.csv  (177 MB)
  ✅ Friday-WorkingHours-Morning.pcap_ISCX.csv  (58 MB)
  ✅ Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  (83 MB)
  ✅ Wednesday-workingHours.pcap_ISCX.csv  (225 MB)

Copied 8 raw CSV files → /content/AI-Network-Intrusion-Detection-System/data/raw


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Verify everything is in place
# ══════════════════════════════════════════════════════════════════════════════
print("═" * 60)
print("  VERIFICATION — all saved artifacts")
print("═" * 60)

total_size = 0
for dirpath, dirnames, filenames in os.walk(ROOT):
    # Skip empty dirs
    if not filenames:
        continue
    rel = dirpath.replace(ROOT, "").lstrip("/")
    print(f"\n  📁 {rel}/")
    for fname in sorted(filenames):
        fpath = os.path.join(dirpath, fname)
        size  = os.path.getsize(fpath)
        total_size += size
        size_str = f"{size/1e6:.1f} MB" if size > 1e5 else f"{size/1e3:.1f} KB"
        print(f"     ✅  {fname:<45s} {size_str}")

print(f"\n{'═'*60}")
print(f"  Total size: {total_size/1e6:.1f} MB")
print("═" * 60)

════════════════════════════════════════════════════════════
  VERIFICATION — all saved artifacts
════════════════════════════════════════════════════════════

  📁 reports/metrics/
     ✅  iso_metrics.csv                               0.2 KB
     ✅  rf_class_report.csv                           1.2 KB
     ✅  rf_metrics.csv                                0.1 KB
     ✅  xgb_class_report.csv                          1.2 KB
     ✅  xgb_metrics.csv                               0.1 KB

  📁 reports/figures/
     ✅  anomaly_score_histogram.png                   41.5 KB
     ✅  attack_distribution.png                       86.2 KB
     ✅  confusion_matrix_rf.png                       0.2 MB
     ✅  confusion_matrix_xgb.png                      0.2 MB
     ✅  feature_importance.png                        76.9 KB

  📁 data/sample/
     ✅  sample_flows.csv                              45.6 KB

  📁 data/raw/
     ✅  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv 77.1 MB
     ✅  Friday-WorkingHo

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Zip & Download
# ══════════════════════════════════════════════════════════════════════════════
import shutil
from google.colab import files

ZIP_PATH = "/content/AI-Network-Intrusion-Detection-System"

print("Zipping project folder...")
shutil.make_archive(
    base_name=ZIP_PATH,   # output: /content/AI-Network-Intrusion-Detection-System.zip
    format="zip",
    root_dir="/content",
    base_dir="AI-Network-Intrusion-Detection-System"
)

zip_size = os.path.getsize(ZIP_PATH + ".zip") / 1e6
print(f"✅ Created: AI-Network-Intrusion-Detection-System.zip  ({zip_size:.1f} MB)")

print("\nStarting download...")
files.download(ZIP_PATH + ".zip")
print("✅ Download triggered.")

Zipping project folder...
✅ Created: AI-Network-Intrusion-Detection-System.zip  (641.7 MB)

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered.
